# Day 1 · 02 · SQL Programming & Analytics Architecture


## Business Scenario

The team can now discover data and run SQL. The next step is to write SQL that
can survive review: named steps, predictable windows, reusable views and clear
architecture decisions. This notebook prepares participants for Workshop 1,
where they build the first durable Gold model.


## Learning Objectives

By the end of this notebook participants can:

- use CTEs to express BI logic as readable steps,
- choose the right window function for ranking, deduplication and trends,
- distinguish TEMP VIEW, VIEW, MATERIALIZED VIEW and table use cases,
- understand SQL Scripting as a validation/orchestration tool,
- explain Medallion next to Kimball, Inmon, Data Vault and Data Mesh,
- describe SCD Type 1 vs Type 2 in terms of Power BI history.


In [ ]:
%run ../../setup/00_setup

### Configuration

Display the active RetailHub catalog context and validate prerequisite objects before the demo starts.


In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE",  BRONZE),
    ("SILVER",  SILVER),
    ("GOLD",    GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Verify Silver tables exist before starting. Gold is not checked here — Workshop 1 builds it later.


In [ ]:
required_tables = [
    f"{SILVER}.customers",
    f"{SILVER}.order_lines",
    f"{SILVER}.sales_orders",
    f"{SILVER}.products",
]

missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing required tables:")
    for t in missing:
        print("  -", t)
    raise Exception("Pre-check failed: missing Silver tables. Run data/generate_training_dataset.ipynb first.")

print("[OK] Pre-check passed — all Silver tables exist:")
for t in required_tables:
    print("  -", t)

## 1. CTEs — Readable SQL Pipelines


A CTE (Common Table Expression) names an intermediate result set. Instead of nested subqueries, you write a linear sequence of named steps — like a recipe.


In [ ]:
%sql
WITH completed_lines AS (
  SELECT order_id, product_id, quantity, unit_price,
         quantity * unit_price AS line_revenue
  FROM silver.order_lines
  WHERE status = 'Completed'
),
customer_revenue AS (
  SELECT o.customer_id, SUM(l.line_revenue) AS total_revenue
  FROM completed_lines l
  JOIN silver.sales_orders o ON l.order_id = o.order_id
  GROUP BY o.customer_id
)
SELECT customer_id, ROUND(total_revenue, 2) AS total_revenue
FROM customer_revenue
ORDER BY total_revenue DESC
LIMIT 10

Multi-CTE pipeline — filter → aggregate → rank → top 10 customers with their rank:


In [ ]:
%sql
WITH completed AS (
  SELECT l.order_id, l.product_id, l.quantity * l.unit_price AS revenue
  FROM silver.order_lines l
  WHERE l.status = 'Completed'
),
by_customer AS (
  SELECT o.customer_id, SUM(c.revenue) AS total_revenue
  FROM completed c
  JOIN silver.sales_orders o ON c.order_id = o.order_id
  GROUP BY o.customer_id
),
ranked AS (
  SELECT customer_id, total_revenue,
         DENSE_RANK() OVER (ORDER BY total_revenue DESC) AS rank
  FROM by_customer
)
SELECT * FROM ranked WHERE rank <= 10

When to use each pattern:

| Pattern | Use when |
|---|---|
| CTE | Multi-step logic, readability, same subquery used twice |
| Subquery | One-off inline filter, simple reference |
| Temp view | Reuse across multiple queries in same session |
| Gold table | Reuse across sessions, BI consumption, scheduled |


## 2. Window Functions


GROUP BY collapses rows into one per group. Window functions add a computed column WITHOUT collapsing rows — each input row gets an output row with the window result.


In [ ]:
%sql
WITH revenue_by_product AS (
  SELECT product_id, category,
         SUM(quantity * unit_price) AS revenue
  FROM silver.order_lines
  WHERE status = 'Completed'
  GROUP BY product_id, category
)
SELECT product_id, category, revenue,
       RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS rank_in_category
FROM revenue_by_product
ORDER BY category, rank_in_category
LIMIT 20

ROW_NUMBER for deduplication — assign a unique row number within a partition:


In [ ]:
%sql
SELECT order_id, line_id, product_id, status,
       ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY line_id) AS line_number
FROM silver.order_lines
LIMIT 20

Running total — SUM OVER accumulates as rows are processed:


In [ ]:
%sql
WITH daily AS (
  SELECT order_date, SUM(quantity * unit_price) AS daily_revenue
  FROM silver.order_lines
  WHERE status = 'Completed'
  GROUP BY order_date
)
SELECT order_date, daily_revenue,
       SUM(daily_revenue) OVER (ORDER BY order_date) AS running_total
FROM daily
ORDER BY order_date

LAG and LEAD — compare current row to previous/next:


In [ ]:
%sql
WITH monthly AS (
  SELECT DATE_FORMAT(order_date, 'yyyy-MM') AS year_month,
         SUM(quantity * unit_price) AS revenue
  FROM silver.order_lines
  WHERE status = 'Completed'
  GROUP BY DATE_FORMAT(order_date, 'yyyy-MM')
)
SELECT year_month, revenue,
       LAG(revenue, 1) OVER (ORDER BY year_month) AS prev_month_revenue,
       ROUND(100.0 * (revenue - LAG(revenue, 1) OVER (ORDER BY year_month))
             / NULLIF(LAG(revenue, 1) OVER (ORDER BY year_month), 0), 1) AS mom_pct
FROM monthly
ORDER BY year_month

| Function | BI use case |
|---|---|
| RANK / DENSE_RANK | Top N products per category, leaderboards |
| ROW_NUMBER | Dedup latest record per customer, pick first per group |
| SUM OVER | Running totals, cumulative budget tracking |
| LAG / LEAD | Month-over-month, day-over-day comparisons |


## 3. Views — from Ephemeral to Permanent


| Type | Persistence | Refresh | Use case |
|---|---|---|---|
| TEMP VIEW | Session only | On query | Intermediate step, dev/test |
| VIEW | Schema-permanent | On query (recomputed) | Shared logic, no storage cost |
| MATERIALIZED VIEW | Schema-permanent | Scheduled / on refresh | Pre-computed aggregates, faster BI |
| STREAMING TABLE | Schema-permanent | Continuous / triggered | Auto Loader, Kafka ingest |
| Gold TABLE | Schema-permanent | Explicit refresh (pipeline) | BI contract, owned by data team |


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW completed_by_channel AS
SELECT channel,
       COUNT(DISTINCT o.order_id) AS orders,
       SUM(l.quantity * l.unit_price) AS revenue
FROM silver.order_lines l
JOIN silver.sales_orders o ON l.order_id = o.order_id
WHERE l.status = 'Completed'
GROUP BY channel

In [ ]:
%sql
SELECT * FROM completed_by_channel ORDER BY revenue DESC

This TEMP VIEW lives only in this session. After cluster restart — it's gone. No storage cost, no schema pollution.


**Materialized View (MV)** — pre-computed and stored. Databricks refreshes it on a schedule or on demand. Use for large aggregates where re-running the full query on every BI request is too expensive.


In [ ]:
try:
    spark.sql("""
        CREATE OR REPLACE MATERIALIZED VIEW silver.mv_demo_channel_revenue AS
        SELECT channel, SUM(quantity * unit_price) AS revenue
        FROM silver.order_lines
        WHERE status = 'Completed'
        GROUP BY channel
    """)
    print("[OK] Materialized View created in silver schema (demo only)")
    print("     In production, MVs live in Gold and are managed by the data team.")
except Exception as e:
    print(f"[INFO] MV not available in this workspace tier: {e}")
    print("       Materialized Views require Databricks SQL Pro or Serverless.")

**Streaming Table** — like a Materialized View but reads from a streaming source (Auto Loader, Kafka). Updated continuously or on trigger. This course does not build streaming tables — your Silver layer was already loaded by the trainer's pipeline.


In [ ]:
%sql
DROP MATERIALIZED VIEW IF EXISTS silver.mv_demo_channel_revenue

## 4. SQL Scripting Fundamentals

SQL Scripting is procedural SQL: variables, `IF`, loops and compound blocks. It
is useful when a repeated analyst check becomes too important for copy/paste but
not yet large enough to become a full engineering pipeline.

| Feature | Use case in BI work |
|---|---|
| `DECLARE` / `SET` | store thresholds, dates or row counts |
| `IF` | fail fast when data is outside expectations |
| `CASE` | define business buckets inside a query |
| loops | repeat profiling across many tables |
| compound block | group validation logic into one repeatable script |

In production, these checks often move into Lakeflow Jobs or DAB-deployed SQL
tasks. Here we show the syntax orientation and keep the notebook robust if the
current runtime does not support compound statements.


In [ ]:
try:
    result = spark.sql("""
    BEGIN ATOMIC
      DECLARE completed_orders BIGINT DEFAULT 0;
      SET completed_orders = (
        SELECT COUNT(DISTINCT order_id) FROM silver.sales_orders WHERE status = 'Completed'
      );
      SELECT completed_orders;
    END
    """)
    display(result)
except Exception as e:
    print('[INFO] SQL Scripting compound statements are not available here.')
    print(type(e).__name__, str(e)[:240])


### SQL Scripting decision rule

Use plain SQL for a single question. Use CTEs when the logic has multiple named
steps. Use SQL Scripting when the logic needs control flow: thresholds, branching
or repeated checks. Use a job when the check must run automatically and fail the
refresh if the contract is broken.


## 5. Medallion Architecture


![Medallion to Power BI](../../assets/images/medallion_to_powerbi.png)


| Layer | Raw name | Responsibility | Quality guarantee |
|---|---|---|---|
| Bronze | `bronze.*` | Land raw data as-is | Schema preserved, no transforms |
| Silver | `silver.*` | Clean, validate, deduplicate | Types correct, nulls handled, grain defined |
| Gold | `gold.*` | Business-ready metrics and dimensions | KPIs agreed, grain documented, BI contract |


In [ ]:
%sql
SHOW TABLES IN silver

In [ ]:
%sql
SHOW TABLES IN gold

Gold is a BI contract: (1) grain — what does one row represent? (2) filter — only Completed orders? (3) KPI owner — who validates the definition? (4) refresh SLA — how stale is acceptable? (5) schema compatibility — Power BI models break if column names change.


## 6. Analytics Architecture Patterns


![Architecture patterns](../../assets/images/architecture_patterns_to_powerbi.png)


**Kimball (Star Schema)** — facts + dimensions, optimized for BI slicing. One fact table (sales, orders, events) surrounded by dimension tables (customer, product, date). Denormalized for fast joins. Default choice for most analytics teams.


**Inmon (Enterprise DW)** — normalized 3NF core warehouse upstream, then data marts (which can be Kimball star schemas) downstream. Suited for large enterprises with many consuming teams and strict data governance. More complex ETL.


**Data Vault** — hubs (business keys), links (relationships), satellites (descriptive attributes). Designed for highly historized, auditable data from many source systems. Complex to query directly — usually has a reporting layer (Kimball mart) on top.


**Data Mesh** is an operating model, not a technical pattern. It defines who owns Gold objects (domain teams, not central IT) and how they publish data products. Can be implemented with any technical pattern above.


| Pattern | BI friendly | Many sources | Audit history | Team size |
|---|---|---|---|---|
| Kimball | ✅ Best | OK | SCD2 only | Small-medium |
| Inmon | OK | ✅ | ✅ | Large enterprise |
| Data Vault | Needs mart | ✅ | ✅ Best | Large, many sources |
| Data Mesh | Depends | ✅ | Depends | Any (governance focus) |


For this course: **Medallion + Kimball**. Gold layer = star schema with dim_date, dim_customer, dim_product + fact_sales + KPI aggregates. This is what you'll build in Workshop 1 and 2.


## 7. Slowly Changing Dimensions (SCD)


Problem: a customer changed their segment from SMB to Enterprise in May 2025. The January 2025 sales report should show SMB (historical truth) or Enterprise (current value)? The answer depends on your SCD strategy.


**SCD Type 1 — Overwrite**. Update the current value, lose history. Use when: correcting a data error (wrong email address), or when historical accuracy doesn't matter.


In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE silver.scd_demo_customer (
        customer_id INT, customer_name STRING, segment STRING
    )
""")
spark.sql("""
    INSERT INTO silver.scd_demo_customer VALUES
    (101, 'Acme Corp', 'SMB'),
    (102, 'Global Inc', 'Enterprise')
""")
print("[OK] Demo table created")

In [ ]:
%sql
UPDATE silver.scd_demo_customer
SET segment = 'Enterprise'
WHERE customer_id = 101

In [ ]:
%sql
SELECT * FROM silver.scd_demo_customer

History is gone. The January report would now show 'Enterprise' for Acme Corp — even though they were SMB in January. For correcting errors: OK. For tracking real business changes: wrong.


**SCD Type 2 — Insert new row with version dates**. Keep history with `valid_from`, `valid_to`, `is_current` columns. History is preserved.


In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE silver.scd2_demo_customer (
        customer_id INT, customer_name STRING, segment STRING,
        valid_from DATE, valid_to DATE, is_current BOOLEAN
    )
""")
spark.sql("""
    INSERT INTO silver.scd2_demo_customer VALUES
    (101, 'Acme Corp', 'SMB',        '2024-01-01', '2025-05-14', false),
    (101, 'Acme Corp', 'Enterprise', '2025-05-15', '9999-12-31', true),
    (102, 'Global Inc','Enterprise', '2024-01-01', '9999-12-31', true)
""")
print("[OK] SCD2 demo table created")

In [ ]:
%sql
SELECT * FROM silver.scd2_demo_customer
WHERE is_current = true

In [ ]:
%sql
SELECT customer_id, customer_name, segment
FROM silver.scd2_demo_customer
WHERE customer_id = 101
  AND '2025-01-15' BETWEEN valid_from AND valid_to

BI impact: Type 1 retrospectively changes the dashboard (January now shows Enterprise). Type 2 shows the correct historical value (January correctly shows SMB). For financial reporting and compliance: Type 2 is required.


In [ ]:
%sql
DROP TABLE IF EXISTS silver.scd_demo_customer;
DROP TABLE IF EXISTS silver.scd2_demo_customer